# 🚀 z/OS MIPS Prediction - Google Colab Pipeline

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chelvy/Perf_Plan/blob/main/notebooks/03_colab_full_pipeline.ipynb)

This notebook runs the complete z/OS MIPS prediction pipeline on Google Colab.

**What this notebook does:**
1. ✅ Clones the repository and installs dependencies
2. ✅ Creates sample data (or uploads your own)
3. ✅ Trains regression and classification models
4. ✅ Evaluates and compares all models
5. ✅ Creates visualizations
6. ✅ Downloads trained models

**Runtime:** ~5-10 minutes (depending on data size)


---
## 📋 Step 0: Setup Environment

First, we'll check if we're running on Colab and set up the environment.

In [ ]:
# Check if running on Colab
try:
    import google.colab
    IN_COLAB = True
    print("✅ Running on Google Colab")
except:
    IN_COLAB = False
    print("⚠️  Not running on Colab")

# Import base libraries
import os
import sys
import subprocess
from pathlib import Path

---
## 📦 Step 1: Clone Repository & Install Dependencies

Choose one of the following options:

### Option A: Clone from GitHub (Recommended)

In [ ]:
%%bash
# Clone the repository
if [ -d "Perf_Plan" ]; then
    echo "Repository already exists, pulling latest changes..."
    cd Perf_Plan && git pull
else
    echo "Cloning repository..."
    git clone https://github.com/chelvy/Perf_Plan.git
fi

In [ ]:
# Change to project directory
if IN_COLAB:
    os.chdir('/content/Perf_Plan')
    
print(f"Current directory: {os.getcwd()}")

### Option B: Install from PyPI (if published)

In [ ]:
# Uncomment if you've published to PyPI
# !pip install zos-mips-prediction

### Install Dependencies

In [ ]:
!pip install -q -r requirements.txt

print("\n✅ All dependencies installed!")

### Verify Installation

In [ ]:
# Import key modules
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add src to path
sys.path.insert(0, 'src')

from src.data_loader import MIPSDataLoader, create_sample_data
from src.preprocessing import MIPSPreprocessor, create_mips_categories
from src.features import MIPSFeatureEngineering
from src.models.regression import RegressionModelFactory, BaselinePredictor
from src.models.classification import ClassificationModelFactory, MajorityClassBaseline
from src.training import MIPSTrainingPipeline
from src.evaluation import ModelEvaluator, ErrorAnalyzer
from src.visualization import MIPSVisualizer

# Settings
pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')
%matplotlib inline

print("\n✅ All modules imported successfully!")

---
## 📊 Step 2: Prepare Data

Choose one of the following options:

### Option A: Create Sample Data (for testing)

In [ ]:
# Create sample data
print("Creating sample z/OS MIPS data...")

sample_path = 'data/sample/sample_mips_data.csv'
df_sample = create_sample_data(
    output_path=sample_path,
    n_samples=2000,      # 2000 records
    n_apps=100,          # 100 different applications
    random_state=42
)

print(f"\n✅ Sample data created: {len(df_sample)} records")
print(f"   Number of applications: {df_sample['application'].nunique()}")
print(f"   Date range: {df_sample['timestamp'].min()} to {df_sample['timestamp'].max()}")

# Set data path for training
DATA_PATH = sample_path

df_sample.head()

### Option B: Upload Your Own Data

In [ ]:
# Uncomment to upload your own CSV file
# if IN_COLAB:
#     from google.colab import files
#     print("Please upload your CSV file with MIPS data...")
#     uploaded = files.upload()
#     
#     # Get the uploaded filename
#     DATA_PATH = list(uploaded.keys())[0]
#     print(f"\n✅ File uploaded: {DATA_PATH}")
# else:
#     DATA_PATH = 'your_data.csv'  # Change this to your file path

### Option C: Connect to Google Drive

In [ ]:
# Uncomment to mount Google Drive
# if IN_COLAB:
#     from google.colab import drive
#     drive.mount('/content/drive')
#     
#     # Set path to your data in Google Drive
#     DATA_PATH = '/content/drive/MyDrive/zos_data/mips_data.csv'
#     print(f"\n✅ Using data from: {DATA_PATH}")

### Quick Data Exploration

In [ ]:
# Load and explore the data
loader = MIPSDataLoader(DATA_PATH)
data = loader.load_csv()

print("\n📊 Data Summary:")
info = loader.get_data_info()
for key, value in info.items():
    if key != 'missing_values':
        print(f"  {key}: {value}")

print("\n📈 Basic Statistics:")
display(data.describe())

print("\n🔍 First few records:")
display(data.head())

---
## 🎨 Step 3: Data Visualization

In [ ]:
# Distribution of MIPS consumption
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].hist(data['MIPS_consumption'], bins=50, edgecolor='black', alpha=0.7, color='green')
axes[0].set_title('MIPS Consumption Distribution', fontweight='bold', fontsize=14)
axes[0].set_xlabel('MIPS Consumption')
axes[0].set_ylabel('Frequency')
axes[0].grid(True, alpha=0.3)

axes[1].boxplot(data['MIPS_consumption'])
axes[1].set_title('MIPS Consumption Box Plot', fontweight='bold', fontsize=14)
axes[1].set_ylabel('MIPS Consumption')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n📊 MIPS Statistics:")
print(f"  Mean: {data['MIPS_consumption'].mean():.2f}")
print(f"  Median: {data['MIPS_consumption'].median():.2f}")
print(f"  Std: {data['MIPS_consumption'].std():.2f}")
print(f"  Range: [{data['MIPS_consumption'].min():.2f}, {data['MIPS_consumption'].max():.2f}]")

In [ ]:
# Correlation heatmap
numeric_cols = data.select_dtypes(include=[np.number]).columns
correlation_matrix = data[numeric_cols].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
            center=0, square=True, linewidths=1, cbar_kws={'shrink': 0.8})
plt.title('Correlation Matrix - MIPS Indicators', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()

---
## 🔧 Step 4: Configure Training Pipeline

In [ ]:
# Training configuration
config = {
    'data_path': DATA_PATH,
    'test_size': 0.2,                    # 20% for testing
    'random_state': 42,                  # For reproducibility
    'time_based_split': False,           # Random split (set True for time-based)
    'scaler_type': 'standard',           # StandardScaler
    'handle_outliers': True,             # Handle outliers
    'outlier_method': 'clip',            # Clip outliers to percentiles
    'feature_engineering': True,         # Apply feature engineering
    'n_categories': 3,                   # For classification: LOW/MEDIUM/HIGH
    'category_method': 'quantile',       # Equal-frequency binning
    'save_models': True,                 # Save trained models
    'output_dir': 'models',             # Model output directory
    'results_dir': 'results'            # Results output directory
}

print("\n⚙️  Training Configuration:")
for key, value in config.items():
    print(f"  {key}: {value}")

---
## 🚀 Step 5: Train Models

This will train all regression and classification models.

In [ ]:
# Create training pipeline
print("\n" + "="*80)
print("🎯 STARTING TRAINING PIPELINE")
print("="*80)

pipeline = MIPSTrainingPipeline(config)

# Run complete pipeline
results = pipeline.run_full_pipeline(mode='both')  # 'regression', 'classification', or 'both'

print("\n" + "="*80)
print("✅ TRAINING COMPLETED SUCCESSFULLY!")
print("="*80)

---
## 📈 Step 6: Analyze Regression Results

In [ ]:
# Compare regression models
if 'regression' in results:
    evaluator = ModelEvaluator()
    
    # RMSE comparison
    rmse_comparison = evaluator.compare_models(
        results['regression'], 
        metric='rmse', 
        mode='regression'
    )
    
    print("\n" + "="*80)
    print("📊 REGRESSION MODELS - RMSE COMPARISON")
    print("="*80)
    display(rmse_comparison.head(15))
    
    # R² comparison
    r2_comparison = evaluator.compare_models(
        results['regression'], 
        metric='r2', 
        mode='regression'
    ).sort_values('test_r2', ascending=False)
    
    print("\n" + "="*80)
    print("📊 REGRESSION MODELS - R² COMPARISON")
    print("="*80)
    display(r2_comparison.head(15))

In [ ]:
# Visualize regression model comparison
if 'regression' in results:
    viz = MIPSVisualizer(output_dir='results')
    
    # Plot RMSE comparison
    viz.plot_model_comparison(
        rmse_comparison.head(15),
        metric='rmse',
        title='Regression Models - RMSE Comparison'
    )

### Best Regression Model Analysis

In [ ]:
# Analyze best regression model
if 'regression' in results and pipeline.best_model_name:
    best_name = pipeline.best_model_name
    best_result = results['regression'][best_name]
    
    print("\n" + "="*80)
    print(f"🏆 BEST REGRESSION MODEL: {best_name}")
    print("="*80)
    
    evaluator.print_regression_summary(
        best_result['test_metrics'], 
        best_name
    )
    
    # Get test predictions
    y_test = results['regression'][best_name]['predictions']['test']
    
    # Note: We need to get y_true from the pipeline
    # For now, we'll reload and split the data
    loader_temp = MIPSDataLoader(DATA_PATH)
    data_temp = loader_temp.load_csv()
    train_temp, test_temp = loader_temp.split_train_test(test_size=0.2, random_state=42)
    _, y_true = loader_temp.get_feature_target_split(test_temp)
    
    # Create visualizations
    viz.plot_predictions_vs_actual(
        y_true, 
        y_test,
        title=f"{best_name} - Predictions vs Actual"
    )
    
    viz.plot_residuals(
        y_true,
        y_test,
        title=f"{best_name} - Residual Analysis"
    )
    
    viz.plot_error_distribution(
        y_true,
        y_test,
        title=f"{best_name} - Error Distribution"
    )

---
## 🎯 Step 7: Analyze Classification Results

In [ ]:
# Compare classification models
if 'classification' in results:
    evaluator = ModelEvaluator()
    
    # Accuracy comparison
    acc_comparison = evaluator.compare_models(
        results['classification'], 
        metric='accuracy', 
        mode='classification'
    )
    
    print("\n" + "="*80)
    print("📊 CLASSIFICATION MODELS - ACCURACY COMPARISON")
    print("="*80)
    print("\nMetric: Accuracy = (1/n) × Σ 𝟙[ŷᵢ = yᵢ]\n")
    display(acc_comparison.head(15))
    
    # F1 comparison
    f1_comparison = evaluator.compare_models(
        results['classification'], 
        metric='f1_macro', 
        mode='classification'
    )
    
    print("\n" + "="*80)
    print("📊 CLASSIFICATION MODELS - F1 SCORE COMPARISON")
    print("="*80)
    display(f1_comparison.head(15))

In [ ]:
# Visualize classification model comparison
if 'classification' in results:
    viz = MIPSVisualizer(output_dir='results')
    
    viz.plot_model_comparison(
        acc_comparison.head(15),
        metric='accuracy',
        title='Classification Models - Accuracy Comparison'
    )

### Best Classification Model Analysis

In [ ]:
# Analyze best classification model
if 'classification' in results:
    best_clf_name = acc_comparison.iloc[0]['model']
    best_clf_result = results['classification'][best_clf_name]
    
    print("\n" + "="*80)
    print(f"🏆 BEST CLASSIFICATION MODEL: {best_clf_name}")
    print("="*80)
    
    evaluator.print_classification_summary(
        best_clf_result['test_metrics'], 
        best_clf_name
    )
    
    # Confusion matrix
    cm = np.array(best_clf_result['test_metrics']['confusion_matrix'])
    classes = ['LOW', 'MEDIUM', 'HIGH'][:config['n_categories']]
    
    viz.plot_confusion_matrix(
        cm,
        classes=classes,
        title=f"{best_clf_name} - Confusion Matrix",
        normalize=False
    )
    
    # Normalized confusion matrix
    viz.plot_confusion_matrix(
        cm,
        classes=classes,
        title=f"{best_clf_name} - Normalized Confusion Matrix",
        normalize=True
    )

---
## 📊 Step 8: Summary & Comparison

In [ ]:
print("\n" + "="*80)
print("📋 FINAL SUMMARY")
print("="*80)

if 'regression' in results:
    print("\n🔵 REGRESSION RESULTS:")
    print(f"   Best Model: {pipeline.best_model_name}")
    best_reg_metrics = results['regression'][pipeline.best_model_name]['test_metrics']
    print(f"   Test RMSE: {best_reg_metrics['rmse']:.2f}")
    print(f"   Test R²: {best_reg_metrics['r2']:.4f}")
    print(f"   Test MAE: {best_reg_metrics['mae']:.2f}")
    print(f"   Test MAPE: {best_reg_metrics['mape']:.2f}%")
    
    # Compare with baseline
    if 'baseline_mean' in results['regression']:
        baseline_rmse = results['regression']['baseline_mean']['test_metrics']['rmse']
        improvement = ((baseline_rmse - best_reg_metrics['rmse']) / baseline_rmse) * 100
        print(f"   \n   📈 Improvement over baseline: {improvement:.1f}%")

if 'classification' in results:
    print("\n🟢 CLASSIFICATION RESULTS:")
    best_clf_name = acc_comparison.iloc[0]['model']
    print(f"   Best Model: {best_clf_name}")
    best_clf_metrics = results['classification'][best_clf_name]['test_metrics']
    print(f"   Test Accuracy: {best_clf_metrics['accuracy']:.4f}")
    print(f"   Test F1 (macro): {best_clf_metrics['f1_macro']:.4f}")
    print(f"   Test Precision (macro): {best_clf_metrics['precision_macro']:.4f}")
    print(f"   Test Recall (macro): {best_clf_metrics['recall_macro']:.4f}")
    
    # Compare with baseline
    if 'baseline_majority' in results['classification']:
        baseline_acc = results['classification']['baseline_majority']['test_metrics']['accuracy']
        improvement = ((best_clf_metrics['accuracy'] - baseline_acc) / baseline_acc) * 100
        print(f"   \n   📈 Improvement over baseline: {improvement:.1f}%")

print("\n" + "="*80)
print("\n✅ All models trained and evaluated successfully!")
print("📁 Models saved to: models/")
print("📁 Results saved to: results/")
print("="*80)

---
## 💾 Step 9: Download Trained Models

Download the trained models and results for use in production.

In [ ]:
# List saved models
print("📦 Saved Models and Files:\n")

if os.path.exists('models'):
    models_files = list(Path('models').glob('*'))
    if models_files:
        print("Models directory:")
        for f in models_files:
            size_mb = f.stat().st_size / (1024 * 1024)
            print(f"  📄 {f.name} ({size_mb:.2f} MB)")
    else:
        print("  No model files found")
else:
    print("  Models directory not found")

if os.path.exists('results'):
    results_files = list(Path('results').glob('*'))
    if results_files:
        print("\nResults directory:")
        for f in results_files:
            size_kb = f.stat().st_size / 1024
            print(f"  📄 {f.name} ({size_kb:.2f} KB)")
    else:
        print("  No result files found")
else:
    print("  Results directory not found")

In [ ]:
# Download files (only works in Colab)
if IN_COLAB:
    from google.colab import files
    import zipfile
    
    print("📦 Creating ZIP archive of models and results...\n")
    
    # Create ZIP file
    zip_filename = 'zos_mips_models_results.zip'
    with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
        # Add models
        if os.path.exists('models'):
            for file in Path('models').rglob('*'):
                if file.is_file():
                    zipf.write(file, arcname=f'models/{file.name}')
        
        # Add results
        if os.path.exists('results'):
            for file in Path('results').rglob('*'):
                if file.is_file():
                    zipf.write(file, arcname=f'results/{file.name}')
    
    print(f"✅ ZIP archive created: {zip_filename}")
    print(f"   Size: {Path(zip_filename).stat().st_size / (1024*1024):.2f} MB\n")
    
    # Download
    print("⬇️  Downloading...")
    files.download(zip_filename)
    print("✅ Download complete!")
else:
    print("⚠️  Not running on Colab - files not downloaded")
    print("📁 Files are saved locally in 'models/' and 'results/' directories")

---
## 🔮 Step 10: Make Predictions (Optional)

Use the trained model to make predictions on new data.

In [ ]:
# Create some new sample data for prediction
print("Creating new data for prediction...\n")

new_data = pd.DataFrame({
    'application': ['APP_001', 'APP_002', 'APP_003', 'APP_004', 'APP_005'],
    'timestamp': pd.date_range('2025-01-01', periods=5, freq='D'),
    'M24H': [1200, 1500, 900, 1800, 1100],
    'MDIU': [1000, 1300, 800, 1600, 950],
    'MPTE': [1400, 1700, 1000, 2000, 1250],
    'TXDIU': [60, 75, 45, 85, 55],
    'EFF': [0.92, 0.88, 0.95, 0.85, 0.90],
    'TVDIU': [120, 140, 100, 160, 110]
})

print("New data for prediction:")
display(new_data)

# Apply feature engineering
feature_eng = MIPSFeatureEngineering()
new_data_fe = feature_eng.create_all_features(new_data)

# Preprocess
X_new = new_data_fe.drop(columns=['timestamp'], errors='ignore')
X_new_scaled = pipeline.preprocessor.transform(X_new)

# Predict with best regression model
if pipeline.best_model:
    predictions = pipeline.best_model.predict(X_new_scaled)
    
    print(f"\n🔮 Predictions from {pipeline.best_model_name}:\n")
    results_df = new_data[['application', 'M24H', 'MDIU', 'MPTE']].copy()
    results_df['Predicted_MIPS'] = predictions
    results_df['Predicted_MIPS'] = results_df['Predicted_MIPS'].round(2)
    
    display(results_df)
else:
    print("⚠️  No trained model available for prediction")

---
## 🎓 Next Steps

Now that you have trained models, you can:

1. **Upload your real z/OS data** (3 years of historical MIPS data)
2. **Fine-tune hyperparameters** for better performance
3. **Deploy models** to production environment
4. **Monitor performance** over time
5. **Retrain periodically** with new data

### Using the models in production:

```python
# Load trained model
from src.models.regression import MIPSRegressionModel
from src.preprocessing import MIPSPreprocessor

model = MIPSRegressionModel.load('models/best_model.pkl')
preprocessor = MIPSPreprocessor.load('models/preprocessor.pkl')

# Prepare new data
X_new_scaled = preprocessor.transform(new_data)

# Predict
predictions = model.predict(X_new_scaled)
```

### Documentation:
- Full documentation in `README.md`
- CLI usage: `python main.py --help`
- Notebook examples in `notebooks/`

---

**Questions or issues?** 
- Check the GitHub repository
- Review the README.md
- Examine the source code in `src/`